# 02. Эксперименты — обучение рекомендателя

Пайплайн: `prepare` → `features` → `train`.

**Финальная модель v7:** LightGBM per-product + hybrid (`min_positives=100`) + early stopping по последнему месяцу train.

| Артефакт | Путь |
|----------|------|
| Модель | `models/model.bin` |
| Метрики | `models/metrics.json` |
| Журнал | `suggestions_04.md` |
| Реестр | `models/experiments_registry.json` |

MLflow (опционально): `python scripts/start_mlflow.py`, эксперимент `bank-product-recommender`.

Ячейки prepare/features/train тяжёлые — по умолчанию достаточно **загрузить уже посчитанные метрики**. Переобучение — опциональные ячейки ниже.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path("..").resolve()
CONFIG_PATH = ROOT / "configs" / "train.yaml"
with CONFIG_PATH.open(encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("ROOT:", ROOT)
print("random_seed:", config["random_seed"])
print("sample train/valid per month:",
      config["data"]["clients_per_month_train"],
      config["data"]["clients_per_month_valid"])
print("min_positives:", config["model"].get("min_positives_train"))
print("early_stopping:", config["model"].get("early_stopping"))
print("experiment:", config["mlflow"]["experiment_name"])
print("train_run:", config["mlflow"].get("train_run_name"))


def run_module(module: str, *extra: str) -> None:
    cmd = [sys.executable, "-m", module, "--config", str(CONFIG_PATH), *extra]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)

ROOT: C:\work\yandex\source\mle-pr-final
random_seed: 42
sample train/valid per month: 100000 133333
min_positives: 100
early_stopping: {'enabled': True, 'holdout_months': 1, 'patience': 20}
experiment: bank-product-recommender
train_run: bank-rec-train-007-early-stopping


## 1. Результаты текущего `models/metrics.json` (v7)

Сравнение с popularity baseline на том же valid.

In [2]:
metrics_path = ROOT / config["paths"]["metrics_path"]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
base = metrics["baseline"]
model = metrics["lightgbm"]
lift = model["map@7"] / base["map@7"] - 1

summary = pd.DataFrame([
    {"model": "popularity", **{k: base[k] for k in ("map@7", "precision@7", "recall@7")}},
    {"model": "lightgbm_v7", **{k: model[k] for k in ("map@7", "precision@7", "recall@7")}},
])
display(summary)
print(f"lift MAP@7 vs baseline: {lift:+.2%}")
print(f"n_train={metrics.get('n_train')}, n_valid={metrics.get('n_valid')}, n_features={metrics.get('n_features')}")
print(f"min_positives={metrics.get('min_positives_train')}, early_stopping={metrics.get('early_stopping')}")
print("skipped → popularity:", metrics.get("skipped_products"))
print("model.bin exists:", (ROOT / config["paths"]["model_path"]).exists())

,model,map@7,precision@7,recall@7
0,popularity,0.020722,0.005681,0.030590
1,lightgbm_v7,0.025657,0.005691,0.030705


lift MAP@7 vs baseline: +23.82%
n_train=1200000, n_valid=533332, n_features=43
min_positives=100, early_stopping={'enabled': True, 'holdout_months': 1, 'patience': 20}
skipped → popularity: ['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cder_fin_ult1', 'ind_ctju_fin_ult1', 'ind_deme_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_viv_fin_ult1']
model.bin exists: True


## 2. AUC по продуктам (eligible-клиенты без продукта на t)

In [3]:
aucs = pd.Series(metrics.get("product_aucs") or {}, dtype="float64").dropna().sort_values(ascending=False)
display(aucs.to_frame("roc_auc").head(15))
iters = {k: v for k, v in (metrics.get("best_iterations") or {}).items() if v is not None}
print("best_iterations (early stopping):", dict(sorted(iters.items(), key=lambda x: -x[1])[:8]), "...")

,roc_auc
ind_nom_pens_ult1,0.958280
ind_nomina_ult1,0.956305
ind_cno_fin_ult1,0.937333
ind_tjcr_fin_ult1,0.932984
ind_ecue_fin_ult1,0.910764
ind_recibo_ult1,0.908611
ind_cco_fin_ult1,0.904398
ind_dela_fin_ult1,0.866098
ind_reca_fin_ult1,0.861537
ind_ctop_fin_ult1,0.860921


best_iterations (early stopping): {'ind_recibo_ult1': 183, 'ind_cco_fin_ult1': 110, 'ind_nom_pens_ult1': 79, 'ind_tjcr_fin_ult1': 78, 'ind_cno_fin_ult1': 65, 'ind_ecue_fin_ult1': 64, 'ind_nomina_ult1': 60, 'ind_dela_fin_ult1': 43} ...


## 3. Краткая история экспериментов

Подробности и код — в `suggestions_04.md`.

In [4]:
reg_path = ROOT / "models" / "experiments_registry.json"
if reg_path.exists():
    reg = json.loads(reg_path.read_text(encoding="utf-8"))
    rows = []
    for r in reg.get("runs", []):
        rows.append({
            "id": r.get("id"),
            "name": r.get("name"),
            "sample": r.get("sample"),
            "map@7": r.get("map@7"),
            "lift_vs_baseline_%": r.get("lift_vs_baseline_pct"),
            "best": r.get("kept_as_best"),
        })
    display(pd.DataFrame(rows))
    print("best_run:", reg.get("best_run"))
else:
    print("нет", reg_path)

,id,name,sample,map@7,lift_vs_baseline_%,best
0,v1,baseline_lgbm,60k/80k,0.025486,NaN,False
1,v2,spw_ratio_cap50,60k/80k,0.023129,NaN,False
2,v3,spw_sqrt_cap50,60k/80k,0.023111,NaN,False
3,v4,portfolio_lags_1_2,60k/80k,0.025406,NaN,False
4,v5,hybrid_min_positives_100,60k/80k,0.025563,21.81,False
5,v6,sample_100k_hybrid,100k/133k,0.025526,23.18,False
6,v7,early_stopping_train_holdout,100k/133k,0.025657,23.81,True
7,user_split_001,user_split_parallel_to_v7,"100k/133k per month, all months in both folds",0.027702,22.31,False
8,v8,bayes_shrink_prior100,100k/133k,0.025659,23.83,False
9,v9,time_sample_weights_hl3,100k/133k,0.025556,23.33,False


best_run: v7


## 4. Результат

1. **Baseline** popularity: MAP@7 ≈ 0.021.
2. **v7** (hybrid + ES + сэмпл 100k): MAP@7 ≈ **0.0257**, lift ≈ **+24%**.
3. SPW и полные лаги портфеля не улучшили MAP@7 — не вошли в финал.
4. Редкие продукты отдаём popularity (`min_positives_train=100`).
5. Сервис: `POST /recommend` читает `model.bin` + `data/serving/clients_features.parquet`.

## 5. (Опционально) Переобучить пайплайн

Раскомментируйте / выполните, если нужно пересобрать данные и модель.

In [5]:
# run_module("src.data.prepare")
# run_module("src.features.build")
# run_module("src.models.train", "--skip-mlflow")  # или без --skip-mlflow при поднятом MLflow
print("Пропуск переобучения — используем готовые models/metrics.json")

Пропуск переобучения — используем готовые models/metrics.json
